# Эксперименты: сравнение моделей на критериальном таргете

Площадка для перебора моделей на одной задаче и одной честной валидации.
Таргет — критериальная перспективность ГИС Интегро (не геохимия), признаки —
независимый набор из `dataset_v1` (28 признаков, без факторных).

Все модели — в `experiment/model.py` с единым интерфейсом
`fit(X, y) → predict_score(X)` и реестром `MODELS`. Сравнение — через общую
`experiment.model.spatial_cv` (GroupKFold по пространственным блокам; ROC-AUC,
PR-AUC, lift@N%).

**Модели в реестре:** логистическая регрессия (elastic-net), LightGBM, XGBoost,
RF+GB (боевой baseline), Extra Trees, kNN, MLP, k-means+калибровка,
GMM+калибровка, карта Кохонена (SOM). Отдельно — **U-Net по растру** (Tier 2,
требует PyTorch), см. последнюю секцию.

In [ ]:
import sys, pathlib, warnings, time
warnings.filterwarnings('ignore')
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'experiment' else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))
import numpy as np
import matplotlib.pyplot as plt
from src.criterial_data import load_criterial_dataset
from experiment import model as M

data = load_criterial_dataset()
print('ячеек:', len(data.y), '| признаков:', len(data.feature_cols),
      '| доля класса:', round(float(data.y.mean()), 3))
print('доступные модели:', M.available_models())

## 1. Сравнение всех моделей (пространственная CV)

Один прогон = одна модель через `spatial_cv`. RF+GB (`rfgb`) — самый медленный
(несколько минут); остальные быстрые. Метрики честные (блочная CV).

In [ ]:
rows = []
for name in M.available_models():
    t = time.time()
    r = M.spatial_cv(name, data.X, data.y, data.groups)
    rows.append((name, r['roc_auc'], r['pr_auc'], r['lift'][0.10], time.time()-t))
    print(f"{name:11s} ROC={r['roc_auc']:.3f}  PR={r['pr_auc']:.3f}  "
          f"lift@10%={r['lift'][0.10]:.2f}  ({time.time()-t:.0f}s)")
rows.sort(key=lambda x: -x[1])

### Рейтинг моделей

In [ ]:
names = [r[0] for r in rows]; rocs = [r[1] for r in rows]; lifts = [r[3] for r in rows]
fig, ax = plt.subplots(1, 2, figsize=(13, 5))
ax[0].barh(names[::-1], rocs[::-1]); ax[0].set_xlim(0.5, 1.0); ax[0].set_title('ROC-AUC (больше — лучше)')
ax[0].axvline(0.5, ls=':', color='gray')
ax[1].barh(names[::-1], lifts[::-1]); ax[1].set_title('lift@10% (во сколько раз выше случайного)')
ax[1].axvline(1.0, ls=':', color='gray')
plt.tight_layout(); plt.show()
print(f"{'model':11s} {'ROC':>6s} {'PR':>6s} {'lift@10%':>9s}")
for nm, roc, pr, lf, _ in rows:
    print(f'{nm:11s} {roc:6.3f} {pr:6.3f} {lf:9.2f}')

## 2. Карта прогноза лучшей модели vs критериальный таргет

In [ ]:
best = rows[0][0]
model = M.MODELS[best]().fit(data.X, data.y)
score = model.predict_score(data.X)
r = data.frame['row'].to_numpy().astype(int); k = data.frame['col'].to_numpy().astype(int)
pred = np.full(data.grid_shape, np.nan); pred[r, k] = score
crit = np.full(data.grid_shape, np.nan); crit[r, k] = data.crit
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
im0 = ax[0].imshow(pred, cmap='magma', origin='upper'); ax[0].set_title(f'Лучшая модель: {best} — P(перспективно)')
fig.colorbar(im0, ax=ax[0], shrink=.7)
im1 = ax[1].imshow(crit, cmap='viridis_r', origin='upper'); ax[1].set_title('Критериальный анализ (таргет)')
fig.colorbar(im1, ax=ax[1], shrink=.7)
plt.tight_layout(); plt.show()

## 3. Корреляция признаков с таргетом

Слева — |корреляция| каждого признака с критериальной перспективностью (Spearman,
т.к. связи нелинейные; знак приведён так, что «+» = помогает перспективности).
Справа — матрица корреляций признак-признак (видны группы коллинеарных
геофизических каналов).

In [ ]:
cor = M.feature_correlations(data, method='spearman')   # (имя, corr_с_крит, corr_с_y)
names = [c[0] for c in cor]; rc = [c[1] for c in cor]
fig, ax = plt.subplots(1, 2, figsize=(15, 7))
order = np.argsort(np.abs(rc))
ax[0].barh([names[i] for i in order], [rc[i] for i in order])
ax[0].axvline(0, color='gray', lw=.8); ax[0].set_title('Корреляция с перспективностью (Spearman)')
Cm = M.feature_corr_matrix(data, method='spearman')
im = ax[1].imshow(Cm.values, cmap='coolwarm', vmin=-1, vmax=1)
ax[1].set_xticks(range(len(Cm))); ax[1].set_xticklabels(Cm.columns, rotation=90, fontsize=6)
ax[1].set_yticks(range(len(Cm))); ax[1].set_yticklabels(Cm.columns, fontsize=6)
ax[1].set_title('Матрица корреляций признак-признак'); fig.colorbar(im, ax=ax[1], shrink=.7)
plt.tight_layout(); plt.show()
print('Топ-8 по |corr| с таргетом:')
for n, r_c, r_y in cor[:8]:
    print(f'  {n:14s} crit={r_c:+.3f}  y={r_y:+.3f}')

## 4. Важность признаков (лучшая модель)

Слева — **permutation importance** (модель-агностик: падение ROC-AUC при
перемешивании признака; работает для любой модели). Справа — **встроенная**
важность модели, если она её даёт (бустинг/леса/линейная). Permutation отражает
вклад в качество, встроенная — как часто признак используется в сплитах.

In [ ]:
best_model = M.MODELS[rows[0][0]]().fit(data.X, data.y)
perm = M.permutation_importance(best_model, data.X, data.y, data.feature_cols, n_repeats=5)
nat = M.native_importance(best_model, data.feature_cols)
fig, ax = plt.subplots(1, 2 if nat else 1, figsize=(15, 6))
ax = np.atleast_1d(ax)
pn = [p[0] for p in perm[:15]][::-1]; pv = [p[1] for p in perm[:15]][::-1]
ax[0].barh(pn, pv); ax[0].set_title(f'Permutation importance — {rows[0][0]} (падение ROC-AUC)')
if nat:
    nn = [n for n, _ in nat[:15]][::-1]; nv = [v for _, v in nat[:15]][::-1]
    ax[1].barh(nn, nv); ax[1].set_title(f'Встроенная важность — {rows[0][0]}')
plt.tight_layout(); plt.show()

## 5. Карта Кохонена: U-matrix и component planes
(интерпретируемая безнадзорная модель; см. также `experiment/model.py`)

In [ ]:
som = M.SomProspectivity(grid=(12, 12), n_epochs=30).fit(data.X, data.y)
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].imshow(som.som.u_matrix(), cmap='bone_r'); ax[0].set_title('U-matrix (границы кластеров)')
for a, nm in zip(ax[1:], ['mask_svita', 'gm_gr_2G_25']):
    j = data.feature_cols.index(nm)
    a.imshow(som.som.W_[:, j].reshape(som.som.rows, som.som.cols), cmap='coolwarm'); a.set_title(f'plane: {nm}')
plt.tight_layout(); plt.show()

## 6. U-Net по растру (Tier 2 — пространственный контекст)

Единственная модель, использующая пространственную структуру листа (текстуру
геофизических полей). Работает не по строкам-ячейкам, а по всему растру
(C×H×W), поэтому у неё своя блочная CV — `experiment.model.spatial_cv_unet`.

**Требует PyTorch** (`pip install torch`). Раскомментируйте для запуска:

In [ ]:
# import experiment.model as M
# res_unet = M.spatial_cv_unet(data, epochs=200, base=32)
# print('U-Net  ROC=%.3f  PR=%.3f  lift@10%%=%.2f' % (
#     res_unet['roc_auc'], res_unet['pr_auc'], res_unet['lift'][0.10]))

## Как добавить свою модель

Реализовать класс с интерфейсом `fit(X, y) → predict_score(X)` в
`experiment/model.py` и вписать в `MODELS` — он сразу попадёт в сравнение:

```python
class MyModel:
    def fit(self, X, y): ...; return self
    def predict_score(self, X): ...  # непрерывная оценка [0..1]

MODELS['mymodel'] = MyModel
```